In [4]:
import cv2
import numpy as np

# Initialize webcam
cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

# HSV ranges for RED color
LOWER_RED1 = np.array([0, 100, 100])
UPPER_RED1 = np.array([10, 255, 255])

LOWER_RED2 = np.array([170, 100, 100])
UPPER_RED2 = np.array([179, 255, 255])

canvas = None
prev_point = None

print("Air Canvas Started! Wave a RED marker in front of the camera.")
print("Press 'c' to clear. Press 'q' to quit.")

while cap.isOpened():
    success, frame = cap.read()

    if not success:
        break

    frame = cv2.flip(frame, 1)

    if canvas is None:
        canvas = np.zeros_like(frame)

    # Convert BGR to HSV
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # Detect red color
    mask1 = cv2.inRange(hsv, LOWER_RED1, UPPER_RED1)
    mask2 = cv2.inRange(hsv, LOWER_RED2, UPPER_RED2)
    mask = cv2.bitwise_or(mask1, mask2)

    # Remove noise
    mask = cv2.erode(mask, None, iterations=1)
    mask = cv2.dilate(mask, None, iterations=1)

    # Find contours
    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    curr_point = None

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)

        if cv2.contourArea(largest_contour) > 500:
            M = cv2.moments(largest_contour)

            if M["m00"] != 0:
                cX = int(M["m10"] / M["m00"])
                cY = int(M["m01"] / M["m00"])

                curr_point = (cX, cY)

                # Yellow tracking point
                cv2.circle(
                    frame,
                    curr_point,
                    8,
                    (0, 255, 255),
                    -1
                )

    # Draw when marker moves
    if curr_point is not None and prev_point is not None:
        cv2.line(
            canvas,
            prev_point,
            curr_point,
            (0, 0, 255),
            5
        )

    prev_point = curr_point

    # Add drawing to webcam image
    combined = cv2.add(frame, canvas)

    # Instructions
    cv2.putText(
        combined,
        "Press 'c' to Clear | Press 'q' to Quit",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.imshow("Virtual Air Canvas - RED Marker", combined)

    key = cv2.waitKey(1) & 0xFF

    if key == ord("c"):
        canvas = np.zeros_like(frame)

    elif key == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

Air Canvas Started! Wave a RED marker in front of the camera.
Press 'c' to clear. Press 'q' to quit.
